# D4 - Building SDMX Queries and Retrieving Data

## Objective

In the previous notebooks, we discovered **what datasets are available**, **how they are structured**, and **which values are valid for each dimension**.

In this notebook, we will use that metadata to construct **valid SDMX query keys** and retrieve real observations from the BIS SDMX API.

By the end of this notebook, we will understand:

- How an SDMX query is constructed.
- How dimensions are ordered in a query.
- How to specify values for each dimension.
- How to request a subset of a dataset.
- How to retrieve observations and convert them into a pandas DataFrame.

This notebook marks the transition from **metadata discovery** to **working with real data**.

---

## Progress So Far

```text
Notebook 1
Discover Dataflows
        │
        ▼
Notebook 2
Explore the Data Structure Definition (DSD)
        │
        ▼
Notebook 3
Build a Metadata Explorer
        │
        ▼
Notebook 4
Build Queries & Retrieve Data
```

---

## Expected Outcome

By the end of this notebook, we will be able to retrieve data such as:

- Construct valid SDMX query keys using the discovered metadata.
- Retrieve observations for a selected subset of a dataset.
- Convert the response into a pandas DataFrame for further analysis.

These are the core steps required to query any SDMX-compliant API.

1. Connect to BIS <br>
        ↓
2. Select a Dataflow <br>
        ↓
3. Retrieve the DSD <br>
        ↓
4. Understand the Query Key  ← Today's focus <br>
        ↓
5. Build the first query <br>
        ↓
6. Retrieve observations <br>
        ↓
7. Convert to DataFrame <br>
        ↓
8. Inspect & summarize <br>

In [1]:
# ----------------------------------------------------
# Imports
# ----------------------------------------------------

import pandas as pd
import sdmx

In [2]:
# ----------------------------------------------------
# Connect to the BIS SDMX API
# ----------------------------------------------------

client = sdmx.Client("BIS")

print("Connected to:", client.source.id)

Connected to: BIS


In [3]:
# ----------------------------------------------------
# Retrieve the Total Credit Dataflow
# ----------------------------------------------------

response = client.dataflow()

flow = response.dataflow["WS_TC"]

print("Dataflow ID :", flow.id)
print("Name        :", flow.name)

Dataflow ID : WS_TC
Name        : Total credit


In [4]:
# ----------------------------------------------------
# Retrieve the Data Structure Definition (DSD)
# ----------------------------------------------------

dsd_response = client.get(
    "datastructure",
    resource_id=flow.structure.id
)

dsd = next(iter(dsd_response.structure.values()))

print(dsd)

BIS_TOTAL_CREDIT


In [5]:
# ----------------------------------------------------
# Explore the StructureMessage
# ----------------------------------------------------

collections = []

for attr in dir(dsd_response):

    if attr.startswith("_"):
        continue

    try:
        value = getattr(dsd_response, attr)

        if hasattr(value, "__len__") and not callable(value):
            collections.append({
                "Collection": attr,
                "Type": type(value).__name__,
                "Count": len(value)
            })

    except Exception:
        pass

df_collections = pd.DataFrame(collections)

df_collections

,Collection,Type,Count
0,categorisation,DictLike,0
1,category_scheme,DictLike,0
2,codelist,DictLike,14
3,concept_scheme,DictLike,1
4,constraint,DictLike,0
5,custom_type_scheme,DictLike,0
6,dataflow,DictLike,1
7,hierarchical_codelist,DictLike,0
8,hierarchy,DictLike,0
9,metadataflow,DictLike,0


In [6]:
# ----------------------------------------------------
# Retrieve all Codelists
# ----------------------------------------------------

codelists = dsd_response.codelist

print(f"Total Codelists : {len(codelists)}")

print("\nAvailable Codelists:\n")

for cid in codelists:
    print(cid)

Total Codelists : 14

Available Codelists:

CL_ADJUST
CL_AREA
CL_BIS_UNIT
CL_COLLECTION
CL_CONF_STATUS
CL_DECIMALS
CL_FREQ
CL_OBS_STATUS
CL_SUB_CHANNEL
P
CL_TC_BORROWERS
CL_TC_LENDERS
CL_UNIT_MULT
CL_VALUATION


In [7]:
# ----------------------------------------------------
# Codelist Explorer
# ----------------------------------------------------

def explore_codelist(codelist_id):
    """
    Display all codes in a BIS SDMX codelist.

    Parameters
    ----------
    codelist_id : str
        Example:
            CL_FREQ
            CL_AREA
            CL_TC_BORROWERS
    """

    codelist = codelists[codelist_id]

    rows = []

    for code in codelist.items.values():

        rows.append({
            "Code": code.id,
            "Description": str(code.name)
        })

    return pd.DataFrame(rows)

In [8]:
# ----------------------------------------------------
# Explore Frequency Codes
# ----------------------------------------------------

explore_codelist("CL_FREQ")

,Code,Description
0,A,Annual
1,B,Daily - business week (not supported)
2,D,Daily
3,E,Event (not supported)
4,H,Half-yearly
5,M,Monthly
6,Q,Quarterly
7,W,Weekly


## Selecting Values for Each Dimension

The DSD defines the **order of dimensions**, while the codelists define the **valid values** that can be used for each dimension.

Before constructing a query, we need to select one valid value for each dimension.

In this section, we will inspect the relevant codelists and choose appropriate values for a sample query.

In [10]:
# ----------------------------------------------------
# Frequency
# ----------------------------------------------------

df_freq = explore_codelist("CL_FREQ")

df_freq

,Code,Description
0,A,Annual
1,B,Daily - business week (not supported)
2,D,Daily
3,E,Event (not supported)
4,H,Half-yearly
5,M,Monthly
6,Q,Quarterly
7,W,Weekly


In [13]:
df_freq[df_freq['Code']=='Q']['Description']

6    Quarterly
Name: Description, dtype: object

In [14]:
# ----------------------------------------------------
# Countries
# ----------------------------------------------------

df_area = explore_codelist("CL_AREA")

df_area.head(20)

,Code,Description
0,1X,ECB
1,4T,Emerging market economies (aggregate)
2,5A,All reporting economies
3,5R,Advanced economies
4,AE,United Arab Emirates
5,AL,Albania
6,AR,Argentina
7,AT,Austria
8,AU,Australia
9,BA,Bosnia and Herzegovina


In [15]:
df_area[df_area["Description"] == "India"]

,Code,Description
47,IN,India


In [16]:
# ----------------------------------------------------
# Borrower Sectors
# ----------------------------------------------------

df_borrowers = explore_codelist("CL_TC_BORROWERS")

df_borrowers

,Code,Description
0,C,Non financial sector
1,G,General government
2,H,Households & NPISHs
3,N,Non-financial corporations
4,P,Private non-financial sector


In [17]:
# ----------------------------------------------------
# Lender Sectors
# ----------------------------------------------------

df_lenders = explore_codelist("CL_TC_LENDERS")

df_lenders

,Code,Description
0,A,All sectors
1,B,"Banks, domestic"


In [18]:
# ----------------------------------------------------
# Valuation Methods
# ----------------------------------------------------

df_valuation = explore_codelist("CL_VALUATION")

df_valuation

,Code,Description
0,M,Market value
1,N,Nominal value


In [19]:
# ----------------------------------------------------
# Unit Types
# ----------------------------------------------------

df_units = explore_codelist("CL_BIS_UNIT")

df_units.head(20)

,Code,Description
0,000,Unknown
1,001,100 - yield
2,002,EUR / MWh
3,003,"Index, 1996 Jan 2 = 100"
4,004,Euro / troy ounce
5,005,US Dollar / troy ounce
6,006,US Dollar / lb
7,007,US Dollar / US gal
8,008,US Dollar / Million British Thermal Unit
9,009,US Dollar / Bushel


In [20]:
# ----------------------------------------------------
# Adjustment Types
# ----------------------------------------------------

df_adjust = explore_codelist("CL_ADJUST")

df_adjust

,Code,Description
0,0,Non seasonally adjusted
1,1,Seasonally adjusted
2,A,Adjusted for breaks
3,U,Unadjusted


In [21]:
# ----------------------------------------------------
# Search for Credit Units
# ----------------------------------------------------

df_units[
    df_units["Description"].str.contains(
        "Domestic currency|USD|Percentage|Index|Credit",
        case=False,
        na=False
    )
]

,Code,Description
3,003,"Index, 1996 Jan 2 = 100"
104,193,"Index, 1914 = 1"
105,194,"Index, 1931 Dec 31 = 100"
106,195,"Index, 1931 Sep 18 = 100"
107,196,"Index, 1937 = 100"
...,...,...
875,962,"Index, 2022 Q2 to 2023 Q1 = 100"
877,964,"Index, 2025 Oct = 100"
878,965,"Index, 2025 = 100"
879,966,"Index, 2026 Jan = 100"


In [22]:
# ----------------------------------------------------
# UNIT_TYPE Dimension
# ----------------------------------------------------

for dim in dsd.dimensions:
    if dim.id == "UNIT_TYPE":
        print(dim.local_representation)

<Representation: CL_BIS_UNIT, [Facet(type=FacetType(is_sequence='false', min_length=3, max_length=3, min_value=None, max_value=None, start_value=None, end_value=None, interval=None, time_interval=None, decimals=None, pattern=None, start_time=None, end_time=None, sentinel_values=None), value=None, value_type=<FacetValueType.string: 1>)]>


In [24]:
# ----------------------------------------------------
# Select Query Parameters
# ----------------------------------------------------

query_parameters = {
    "FREQ": "Q",              # Quarterly
    "BORROWERS_CTY": "IN",    # India
    "TC_BORROWERS": "P",      # Private non-financial sector
    "TC_LENDERS": "A",        # All lenders
    "VALUATION": "M",         # Market value
    "UNIT_TYPE": "???",       # To be determined
    "TC_ADJUST": "0",         # Non-seasonally adjusted
    "TIME_PERIOD": "2023"     # Year
}

pd.DataFrame(
    query_parameters.items(),
    columns=["Dimension", "Selected Code"]
)

,Dimension,Selected Code
0,FREQ,Q
1,BORROWERS_CTY,IN
2,TC_BORROWERS,P
3,TC_LENDERS,A
4,VALUATION,M
5,UNIT_TYPE,???
6,TC_ADJUST,0
7,TIME_PERIOD,2023


In [25]:
# ----------------------------------------------------
# Build the SDMX Query Key
# ----------------------------------------------------

query_key = ".".join(query_parameters.values())

print(query_key)

Q.IN.P.A.M.???.0.2023


## Understanding the SDMX Query Key

The SDMX standard identifies a dataset using a **query key**.

A query key is an ordered list of dimension values separated by a period (`.`).

For the **Total Credit** dataset, the dimensions are:

```
FREQ.BORROWERS_CTY.TC_BORROWERS.TC_LENDERS.VALUATION.UNIT_TYPE.TC_ADJUST.TIME_PERIOD
```

After selecting one valid value for each dimension, the query becomes:

```
Q.IN.P.A.M.?.0.2023
```

Each value must appear **in the exact order defined by the DSD**.

This is part of the SDMX standard and is independent of any programming language or client library.

In [26]:
# ----------------------------------------------------
# Query Components
# ----------------------------------------------------

df_query = pd.DataFrame({
    "Dimension": query_parameters.keys(),
    "Selected Code": query_parameters.values()
})

df_query

,Dimension,Selected Code
0,FREQ,Q
1,BORROWERS_CTY,IN
2,TC_BORROWERS,P
3,TC_LENDERS,A
4,VALUATION,M
5,UNIT_TYPE,???
6,TC_ADJUST,0
7,TIME_PERIOD,2023


In [27]:
# ----------------------------------------------------
# BIS REST API Endpoint
# ----------------------------------------------------

print("Dataflow :", flow.id)
print("Query Key:", query_key)

Dataflow : WS_TC
Query Key: Q.IN.P.A.M.???.0.2023


### How the BIS API Uses the Query Key

The BIS SDMX REST API expects the query key to be passed as part of the request URL.

Conceptually, the request looks like:

```
.../data/{dataflow}/{query_key}
```

where:

- `dataflow` identifies the dataset.
- `query_key` specifies the values for each dimension.

Fortunately, the Python `sdmx1` library constructs this request for us, so we rarely need to build the URL manually.

## From the SDMX Standard to the Python Library

So far, we have learned how an SDMX query is constructed according to the SDMX standard.

The Python `sdmx1` library builds on top of this standard and provides a convenient interface for retrieving data without manually constructing HTTP requests.

In the next section, we will use the query we designed to retrieve observations from the BIS SDMX API.

Our Query <br>
      │ <br>
      ▼
Python Library <br>
      │<br>
      ▼
BIS API <br>
      │<br>
      ▼
SDMX Response <br>
      │<br>
      ▼<br>
DataFrame

In [30]:
# ----------------------------------------------------
# Explore the SDMX Client
# ----------------------------------------------------

methods = []

for attr in dir(client):
    if attr.startswith("_"):
        continue

    try:
        value = getattr(client, attr)

        methods.append({
            "Method": attr,
            "Type": type(value).__name__,
            "Callable": callable(value)
        })
    except Exception:
        pass

df_client = pd.DataFrame(methods)

df_client.head(20)

,Method,Type,Callable
0,actualconstraint,partial,True
1,agencyscheme,partial,True
2,allowedconstraint,partial,True
3,attachementconstraint,partial,True
4,availableconstraint,partial,True
5,cache,dict,False
6,categorisation,partial,True
7,categoryscheme,partial,True
8,clear_cache,method,True
9,codelist,partial,True


## Retrieving Data with `client.data()`

The SDMX standard defines **what** a query looks like.

The Python `sdmx1` library provides the `client.data()` method, which sends that query to the SDMX provider and returns an SDMX Data Message.

Instead of manually constructing HTTP requests, we simply specify:

- the dataflow
- the query key
- any additional parameters

The library handles the REST API request internally.

In [31]:
# ----------------------------------------------------
# Inspect client.data()
# ----------------------------------------------------

help(client.data)

Help on partial in module functools:

functools.partial(<bound method Client.get of <s...object at 0x14099b4d0>>, <Resource.data: 'data'>)
    Retrieve SDMX data or metadata with resource_type='data'.

    (Meta)data is retrieved from the :attr:`source` of the current Client. The
    item(s) to retrieve can be specified in one of two ways:

    1. `resource_type`, `resource_id`: These give the type (see :class:`Resource`)
       and, optionally, ID of the item(s). If the `resource_id` is not given, all
       items of the given type are retrieved.
    2. a `resource` object, i.e. a :class:`.MaintainableArtefact`: `resource_type`
       and `resource_id` are determined by the object's class and
       :attr:`id <.IdentifiableArtefact.id>` attribute, respectively.

    Data is retrieved with `resource_type='data'`. In this case, the optional
    keyword argument `key` can be used to constrain the data that is retrieved.
    Examples of the formats for `key`:

    1. ``{'GEO': ['EL', 'ES'

In [32]:
import inspect

print(inspect.signature(client.data))

(resource_id: str | None = None, tofile: 'os.PathLike | IO | None' = None, use_cache: bool = False, dry_run: bool = False, **kwargs) -> 'sdmx.message.Message | requests.Request'


In [35]:
# ----------------------------------------------------
# Retrieve Data Using the SDMX Query Key
# ----------------------------------------------------

response = client.data(
    resource_id=flow.id,
    key=query_key
)

print(type(response))

xml.Reader got no structure=… argument for StructureSpecificData


<class 'sdmx.message.DataMessage'>


In [36]:
print(response)

<sdmx.DataMessage>
  <Header>
    extracted: '2026-07-06T16:10:39'
    id: 'IDREFb8420bc4-1613-4d37-8324-e0d4095dc189'
    prepared: '2026-07-06T16:10:39+00:00'
    reporting_begin: '1951-04-01T00:00:00'
    reporting_end: '2025-10-01T23:59:59'
    receiver: <Agency guest>
    sender: <Agency UNKNOWN>
    source: 
    test: False
  response: <Response [200]>
  DataSet (1)
  dataflow: <DataflowDefinition (missing id)>
  observation_dimension: <Dimension TIME_PERIOD>


In [37]:
dir(response)

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__post_init__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'compare',
 'data',
 'dataflow',
 'footer',
 'header',
 'observation_dimension',
 'response',
 'structure',
 'structure_type',
 'update',
 'version']

In [38]:
response = client.data(
    resource_id=flow.id,
    key=query_key
)

print(type(response))

xml.Reader got no structure=… argument for StructureSpecificData


<class 'sdmx.message.DataMessage'>


In [39]:
# ----------------------------------------------------
# Explore the DataMessage
# ----------------------------------------------------

rows = []

for attr in dir(response):

    if attr.startswith("_"):
        continue

    try:
        value = getattr(response, attr)

        if callable(value):
            continue

        rows.append({
            "Attribute": attr,
            "Type": type(value).__name__
        })

    except Exception:
        pass

pd.DataFrame(rows)

,Attribute,Type
0,data,list
1,dataflow,DataflowDefinition
2,footer,NoneType
3,header,Header
4,observation_dimension,Dimension
5,response,OriginalResponse
6,structure,DataStructureDefinition
7,version,Version


In [40]:
# ----------------------------------------------------
# Inspect DataMessage Collections
# ----------------------------------------------------

collections = []

for attr in dir(response):

    if attr.startswith("_"):
        continue

    try:
        value = getattr(response, attr)

        if callable(value):
            continue

        if hasattr(value, "__len__"):

            collections.append({
                "Collection": attr,
                "Type": type(value).__name__,
                "Length": len(value)
            })

    except Exception:
        pass

pd.DataFrame(collections)

,Collection,Type,Length
0,data,list,1


In [41]:
# ----------------------------------------------------
# Access the Dataset
# ----------------------------------------------------

dataset = response.data[0]

print(type(dataset))
print(dataset)

<class 'sdmx.model.v21.StructureSpecificDataSet'>
<DataSet structured_by=<DataStructureDefinition BIS:WS_TC(2.0)> with 1173 observations>


In [42]:
# ----------------------------------------------------
# Explore the Dataset
# ----------------------------------------------------

rows = []

for attr in dir(dataset):

    if attr.startswith("_"):
        continue

    try:
        value = getattr(dataset, attr)

        if callable(value):
            continue

        rows.append({
            "Attribute": attr,
            "Type": type(value).__name__
        })

    except Exception:
        pass

pd.DataFrame(rows)

,Attribute,Type
0,action,NoneType
1,annotations,list
2,attrib,DictLike
3,described_by,NoneType
4,group,DictLike
5,obs,list
6,series,DictLike
7,structured_by,DataStructureDefinition
8,valid_from,NoneType


In [43]:
dataset

StructureSpecificDataSet(annotations=[], action=None, valid_from=None, described_by=None, structured_by=<DataStructureDefinition BIS:WS_TC(2.0)>, obs=[Observation(attached_attribute={'OBS_STATUS': <AttributeValue: OBS_STATUS=A>, 'OBS_CONF': <AttributeValue: OBS_CONF=F>}, series_key=<SeriesKey: BORROWERS_CTY=IN, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=USD, TC_ADJUST=A, FREQ=Q, TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market value - US dollar - Adjusted for breaks, UNIT_MULT=9, UNIT_MEASURE=USD>, dimension=<Key: TIME_PERIOD=1957-Q1>, value='6.371', group_keys={GroupKey(attrib={'DECIMALS': <AttributeValue: DECIMALS=3>}, described_by=None, values={'BORROWERS_CTY': <KeyValue: BORROWERS_CTY=IN>, 'TC_BORROWERS': <KeyValue: TC_BORROWERS=P>, 'TC_LENDERS': <KeyValue: TC_LENDERS=A>, 'VALUATION': <KeyValue: VALUATION=M>, 'UNIT_TYPE': <KeyValue: UNIT_TYPE=USD>, 'TC_ADJUST': <KeyValue: TC_ADJUST=A>}, id=None)}, value_for=<PrimaryMeasure OBS_VALUE>), Ob

In [44]:
# ----------------------------------------------------
# Inspect Dataset Collections
# ----------------------------------------------------

collections = []

for attr in dir(dataset):

    if attr.startswith("_"):
        continue

    try:
        value = getattr(dataset, attr)

        if callable(value):
            continue

        if hasattr(value, "__len__"):

            collections.append({
                "Collection": attr,
                "Type": type(value).__name__,
                "Length": len(value)
            })

    except Exception:
        pass

pd.DataFrame(collections)

,Collection,Type,Length
0,annotations,list,0
1,attrib,DictLike,0
2,group,DictLike,4
3,obs,list,1173
4,series,DictLike,4


In [45]:
# ----------------------------------------------------
# Inspect the Series
# ----------------------------------------------------

print(f"Total Series: {len(dataset.series)}")

for key in dataset.series.keys():
    print(key)

Total Series: 4
(BORROWERS_CTY=IN, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=USD, TC_ADJUST=A, FREQ=Q, TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market value - US dollar - Adjusted for breaks, UNIT_MULT=9, UNIT_MEASURE=USD)
(BORROWERS_CTY=IN, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=XDC, TC_ADJUST=A, FREQ=Q, TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market value - domestic currency - Adjusted for breaks, UNIT_MULT=9, UNIT_MEASURE=INR)
(BORROWERS_CTY=IN, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=770, TC_ADJUST=A, FREQ=Q, TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market value - Percentage of GDP - Adjusted for breaks, UNIT_MULT=0, UNIT_MEASURE=367)
(BORROWERS_CTY=IN, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=XDC, TC_ADJUST=U, FREQ=Q, TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market value - domestic currency - 

## Understanding Why Multiple Series Were Returned

Although we constrained several dimensions in our query, the response contains **four separate time series**.

This is because the query did not uniquely identify a single series. Some dimensions (such as `UNIT_TYPE` and `TC_ADJUST`) still had multiple valid values that matched the request.

Each unique combination of dimension values defines a different **time series**.

This illustrates an important SDMX concept:

> A dataset is composed of one or more **series**, and each series is identified by a unique combination of dimension values.

In [46]:
# ----------------------------------------------------
# Explore the First Series
# ----------------------------------------------------

first_key = next(iter(dataset.series))

series = dataset.series[first_key]

print(type(series))
print(series)

<class 'list'>
[Observation(attached_attribute={'OBS_STATUS': <AttributeValue: OBS_STATUS=A>, 'OBS_CONF': <AttributeValue: OBS_CONF=F>}, series_key=<SeriesKey: BORROWERS_CTY=IN, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=USD, TC_ADJUST=A, FREQ=Q, TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market value - US dollar - Adjusted for breaks, UNIT_MULT=9, UNIT_MEASURE=USD>, dimension=<Key: TIME_PERIOD=1957-Q1>, value='6.371', group_keys={GroupKey(attrib={'DECIMALS': <AttributeValue: DECIMALS=3>}, described_by=None, values={'BORROWERS_CTY': <KeyValue: BORROWERS_CTY=IN>, 'TC_BORROWERS': <KeyValue: TC_BORROWERS=P>, 'TC_LENDERS': <KeyValue: TC_LENDERS=A>, 'VALUATION': <KeyValue: VALUATION=M>, 'UNIT_TYPE': <KeyValue: UNIT_TYPE=USD>, 'TC_ADJUST': <KeyValue: TC_ADJUST=A>}, id=None)}, value_for=<PrimaryMeasure OBS_VALUE>), Observation(attached_attribute={'OBS_STATUS': <AttributeValue: OBS_STATUS=A>, 'OBS_CONF': <AttributeValue: OBS_CONF=F>}, series_key=<Seri

In [48]:
# ----------------------------------------------------
# Inspect the First Observation
# ----------------------------------------------------

first_obs = series[0]

print(type(first_obs))
print(first_obs)

<class 'sdmx.model.v21.Observation'>
(BORROWERS_CTY=IN, TC_BORROWERS=P, TC_LENDERS=A, VALUATION=M, UNIT_TYPE=USD, TC_ADJUST=A, FREQ=Q, TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market value - US dollar - Adjusted for breaks, UNIT_MULT=9, UNIT_MEASURE=USD, TIME_PERIOD=1957-Q1): 6.371


In [49]:
# ----------------------------------------------------
# Explore the Observation Object
# ----------------------------------------------------

rows = []

for attr in dir(first_obs):

    if attr.startswith("_"):
        continue

    try:
        value = getattr(first_obs, attr)

        if callable(value):
            continue

        rows.append({
            "Attribute": attr,
            "Type": type(value).__name__
        })

    except Exception:
        pass

pd.DataFrame(rows)

,Attribute,Type
0,attached_attribute,DictLike
1,attrib,DictLike
2,dim,Key
3,dimension,Key
4,group_keys,set
5,key,SeriesKey
6,series_key,SeriesKey
7,value,str
8,value_for,PrimaryMeasure


In [51]:
# ----------------------------------------------------
# Inspect the SeriesKey
# ----------------------------------------------------

series_key = next(iter(dataset.series.keys()))

print(type(series_key))
print()

print(series_key.values)
print()

print(type(series_key.values))

print()

print(series_key.values.items())

<class 'sdmx.model.common.SeriesKey'>

{'BORROWERS_CTY': <KeyValue: BORROWERS_CTY=IN>, 'TC_BORROWERS': <KeyValue: TC_BORROWERS=P>, 'TC_LENDERS': <KeyValue: TC_LENDERS=A>, 'VALUATION': <KeyValue: VALUATION=M>, 'UNIT_TYPE': <KeyValue: UNIT_TYPE=USD>, 'TC_ADJUST': <KeyValue: TC_ADJUST=A>, 'FREQ': <KeyValue: FREQ=Q>, 'TITLE_TS': <KeyValue: TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market value - US dollar - Adjusted for breaks>, 'UNIT_MULT': <KeyValue: UNIT_MULT=9>, 'UNIT_MEASURE': <KeyValue: UNIT_MEASURE=USD>}

<class 'sdmx.dictlike.DictLike'>

dict_items([('BORROWERS_CTY', <KeyValue: BORROWERS_CTY=IN>), ('TC_BORROWERS', <KeyValue: TC_BORROWERS=P>), ('TC_LENDERS', <KeyValue: TC_LENDERS=A>), ('VALUATION', <KeyValue: VALUATION=M>), ('UNIT_TYPE', <KeyValue: UNIT_TYPE=USD>), ('TC_ADJUST', <KeyValue: TC_ADJUST=A>), ('FREQ', <KeyValue: FREQ=Q>), ('TITLE_TS', <KeyValue: TITLE_TS=India - Credit to Private non-financial sector from All sectors at Market value - US

In [52]:
for k, v in series_key.values.items():
    print(type(k), k)
    print(type(v), v)
    print("-" * 50)

<class 'str'> BORROWERS_CTY
<class 'sdmx.model.common.KeyValue'> BORROWERS_CTY=IN
--------------------------------------------------
<class 'str'> TC_BORROWERS
<class 'sdmx.model.common.KeyValue'> TC_BORROWERS=P
--------------------------------------------------
<class 'str'> TC_LENDERS
<class 'sdmx.model.common.KeyValue'> TC_LENDERS=A
--------------------------------------------------
<class 'str'> VALUATION
<class 'sdmx.model.common.KeyValue'> VALUATION=M
--------------------------------------------------
<class 'str'> UNIT_TYPE
<class 'sdmx.model.common.KeyValue'> UNIT_TYPE=USD
--------------------------------------------------
<class 'str'> TC_ADJUST
<class 'sdmx.model.common.KeyValue'> TC_ADJUST=A
--------------------------------------------------
<class 'str'> FREQ
<class 'sdmx.model.common.KeyValue'> FREQ=Q
--------------------------------------------------
<class 'str'> TITLE_TS
<class 'sdmx.model.common.KeyValue'> TITLE_TS=India - Credit to Private non-financial sector from Al

In [53]:
# ----------------------------------------------------
# Convert Dataset to a DataFrame
# ----------------------------------------------------

rows = []

for series_key, observations in dataset.series.items():

    # Extract the series metadata
    series_metadata = {
        key: value.value
        for key, value in series_key.values.items()
    }

    # Add one row per observation
    for obs in observations:

        row = series_metadata.copy()

        row["TIME_PERIOD"] = obs.dimension["TIME_PERIOD"].value
        row["OBS_VALUE"] = obs.value

        rows.append(row)

df = pd.DataFrame(rows)

df.head()

,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,FREQ,TITLE_TS,UNIT_MULT,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE
0,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q1,6.371
1,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q2,6.602
2,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q3,6.482
3,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q4,6.69
4,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1958-Q1,7.095


In [54]:
# ----------------------------------------------------
# Dataset Summary
# ----------------------------------------------------

print(f"Number of Observations : {len(df):,}")
print(f"Number of Columns      : {len(df.columns)}")
print()

print("Columns:")
print(df.columns.tolist())

Number of Observations : 1,173
Number of Columns      : 12

Columns:
['BORROWERS_CTY', 'TC_BORROWERS', 'TC_LENDERS', 'VALUATION', 'UNIT_TYPE', 'TC_ADJUST', 'FREQ', 'TITLE_TS', 'UNIT_MULT', 'UNIT_MEASURE', 'TIME_PERIOD', 'OBS_VALUE']


In [55]:
# ----------------------------------------------------
# Convert Data Types
# ----------------------------------------------------

df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1173 entries, 0 to 1172
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   BORROWERS_CTY  1173 non-null   object 
 1   TC_BORROWERS   1173 non-null   object 
 2   TC_LENDERS     1173 non-null   object 
 3   VALUATION      1173 non-null   object 
 4   UNIT_TYPE      1173 non-null   object 
 5   TC_ADJUST      1173 non-null   object 
 6   FREQ           1173 non-null   object 
 7   TITLE_TS       1173 non-null   object 
 8   UNIT_MULT      1173 non-null   object 
 9   UNIT_MEASURE   1173 non-null   object 
 10  TIME_PERIOD    1173 non-null   object 
 11  OBS_VALUE      1173 non-null   float64
dtypes: float64(1), object(11)
memory usage: 110.1+ KB


In [56]:
# ----------------------------------------------------
# Summary Statistics
# ----------------------------------------------------

df.describe(include="all")

,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,FREQ,TITLE_TS,UNIT_MULT,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE
count,1173,1173,1173,1173,1173,1173,1173,1173,1173,1173,1173,1173.000000
unique,1,1,1,1,3,2,1,4,2,3,299,NaN
top,IN,P,A,M,XDC,A,Q,India - Credit to Private non-financial sector...,9,INR,1957-Q1,NaN
freq,1173,1173,1173,1173,598,874,1173,299,874,598,4,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20115.271015
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56646.744011
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.371000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.600000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,114.722000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2523.313000


In [58]:
# ----------------------------------------------------
# Unique Series Returned
# ----------------------------------------------------

series_columns = [
    "UNIT_TYPE",
    "TC_ADJUST"
]

df[series_columns].drop_duplicates().reset_index(drop=True)

,UNIT_TYPE,TC_ADJUST
0,USD,A
1,XDC,A
2,770,A
3,XDC,U


In [59]:
# ----------------------------------------------------
# Example: Domestic Currency Series
# ----------------------------------------------------

df_inr = df[
    (df["UNIT_TYPE"] == "XDC") &
    (df["TC_ADJUST"] == "A")
]

df_inr.head()

,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,FREQ,TITLE_TS,UNIT_MULT,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE
276,IN,P,A,M,XDC,A,Q,India - Credit to Private non-financial sector...,9,INR,1951-Q2,24.279
277,IN,P,A,M,XDC,A,Q,India - Credit to Private non-financial sector...,9,INR,1951-Q3,22.894
278,IN,P,A,M,XDC,A,Q,India - Credit to Private non-financial sector...,9,INR,1951-Q4,22.638
279,IN,P,A,M,XDC,A,Q,India - Credit to Private non-financial sector...,9,INR,1952-Q1,22.777
280,IN,P,A,M,XDC,A,Q,India - Credit to Private non-financial sector...,9,INR,1952-Q2,22.670


In [60]:
# ----------------------------------------------------
# Observations by Series
# ----------------------------------------------------

(
    df.groupby(["UNIT_TYPE", "TC_ADJUST"])
      .size()
      .reset_index(name="Observations")
)

,UNIT_TYPE,TC_ADJUST,Observations
0,770,A,299
1,USD,A,276
2,XDC,A,299
3,XDC,U,299


In [61]:
# ----------------------------------------------------
# Time Coverage
# ----------------------------------------------------

print("First Period :", df["TIME_PERIOD"].min())
print("Last Period  :", df["TIME_PERIOD"].max())

First Period : 1951-Q2
Last Period  : 2025-Q4


In [62]:
# ----------------------------------------------------
# Available Unit Types
# ----------------------------------------------------

df[["UNIT_TYPE", "UNIT_MEASURE"]].drop_duplicates().reset_index(drop=True)

,UNIT_TYPE,UNIT_MEASURE
0,USD,USD
1,XDC,INR
2,770,367


# Summary

In this notebook, we transitioned from **metadata exploration** to **retrieving real SDMX data**.

We learned how to:

- Build a valid SDMX query using the Data Structure Definition (DSD).
- Retrieve observations using the `sdmx1` Python library.
- Understand the structure of an SDMX `DataMessage`.
- Explore how datasets are organized into **Series** and **Observations**.
- Convert the hierarchical SDMX response into a tidy pandas DataFrame.

At this point, we have successfully completed the full SDMX data retrieval workflow:

```text
Dataflow
    ↓
DSD
    ↓
Codelists
    ↓
Query
    ↓
DataMessage
    ↓
Dataset
    ↓
Series
    ↓
Observations
    ↓
pandas DataFrame
```

The resulting DataFrame can now be filtered, analyzed, visualized, and integrated into downstream analytics workflows.

---

